# TalkingData AdTracking Fraud Detection Pipeline (Deep Neural Network - DNN)
**Proyek Ujian Akhir Semester Pembelajaran Mendalam (Deep Learning) - TA 2026/2027**

**Anggota Kelompok:**
1. **231110755 - Ricky Lim** (Ketua / Data Preprocessing & Pipeline Lead)
2. **231110366 - Jericho Valencio alvin** (EDA & Data Pipeline Analyst)
3. **231110715 - Javier Farrel Sim** (DNN Model Architect & Evaluator)
4. **231110103 - Muhammad Hafizul Amri** (Documentation Specialist)
5. **231112208 - Jimmy Makesa Sitepu** (Experiment & Testing Analyst)

---
## Alur Kerja Pipeline ML/DL:
1. **Pemeriksaan & Preprocessing Data**: Optimasi memori & pembuatan fitur frekuensi (*Count Encoding*).
2. **Pemisahan Data & Stratified Resampling**: Pemisahan Train/Test set steril (80:20) & SMOTE pada Training Set.
3. **Pemodelan Deep Neural Network (DNN)**: Jaringan saraf berlapis (*Dense*, *BatchNormalization*, *Dropout*, *ReLU*, *Sigmoid*).
4. **Fitur Unik 1 - Custom Focal Loss**: Penanganan *Extreme Class Imbalance*.
5. **Fitur Unik 2 - Confidence Thresholding**: Adaptasi probabilitas untuk menekan *False Positive*.
6. **Evaluasi Komprehensif & Export**: Metrik Precision, Recall, F1-Score, ROC-AUC, dan penyimpnan artefak `.keras` & `.pkl`.

## 1. Import Library & Konfigurasi Lingkungan

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')
import os

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE

# Konfigurasi Tampilan Grafik
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
print("TensorFlow Version:", tf.__version__)
print("Library berhasil di-import.")

## 2. Load Dataset & Preprocessing Awal (Optimasi Memori)

In [ ]:
TARGET = 'is_attributed'
BASE_FEATURES = ['ip', 'app', 'device', 'os', 'channel']
DTYPES = {
    'ip': 'int32',
    'app': 'int16',
    'device': 'int16',
    'os': 'int16',
    'channel': 'int16',
    'is_attributed': 'int8'
}

# Menggenerasi / Memuat dataset (500.000 sampel)
np.random.seed(42)
n_samples = 500000
df = pd.DataFrame({
    'ip': np.random.randint(1, 100000, n_samples, dtype='int32'),
    'app': np.random.randint(1, 200, n_samples, dtype='int16'),
    'device': np.random.choice([1, 2, 3032, 0], size=n_samples, p=[0.9, 0.05, 0.03, 0.02]).astype('int16'),
    'os': np.random.randint(1, 100, n_samples, dtype='int16'),
    'channel': np.random.randint(1, 500, n_samples, dtype='int16'),
    'is_attributed': np.random.choice([0, 1], size=n_samples, p=[0.997, 0.003]).astype('int8')
})

print("Statistik Singkat Dataset:")
print(df.info())
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribusi Target
sns.countplot(data=df, x=TARGET, ax=axes[0], palette=['#1f77b4', '#d62728'])
axes[0].set_title('Distribusi Class Target (is_attributed)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Normal (0)', 'Fraud/Downloaded (1)'])
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,} ({p.get_height()/len(df)*100:.2f}%)',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')

# Top 10 App ID
top_apps = df['app'].value_counts().head(10)
sns.barplot(x=top_apps.index, y=top_apps.values, ax=axes[1], palette='viridis')
axes[1].set_title('Top 10 App ID Berdasarkan Jumlah Klik', fontsize=12, fontweight='bold')
axes[1].set_xlabel('App ID')
axes[1].set_ylabel('Jumlah Klik')

plt.tight_layout()
plt.show()

## 4. Feature Engineering (Count Encoding)

In [ ]:
print("Membuat Fitur Agregasi Frekuensi (Count Encoding)...")
df['ip_count'] = df.groupby('ip')['app'].transform('count').astype('int32')
df['ip_app_count'] = df.groupby(['ip', 'app'])['device'].transform('count').astype('int32')
df['ip_os_count'] = df.groupby(['ip', 'os'])['device'].transform('count').astype('int32')
df['ip_device_count'] = df.groupby(['ip', 'device'])['app'].transform('count').astype('int32')

FEATURE_COLS = BASE_FEATURES + ['ip_count', 'ip_app_count', 'ip_os_count', 'ip_device_count']
print(f"Total Fitur Input: {len(FEATURE_COLS)} fitur")
df[FEATURE_COLS + [TARGET]].head()

## 5. Pemisahan Data & Resampling (SMOTE)

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET]

# 1. Train-Test Split (80:20) Mencegah Data Leakage
X_train_raw, X_test_raw, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Undersampling/SMOTE pada Training Set
train_df = pd.concat([X_train_raw, y_train_raw], axis=1)
fraud_samples = train_df[train_df[TARGET] == 1]
normal_samples = train_df[train_df[TARGET] == 0]

normal_undersampled = normal_samples.sample(n=len(fraud_samples), random_state=42)
train_balanced = pd.concat([fraud_samples, normal_undersampled]).sample(frac=1, random_state=42).reset_index(drop=True)

X_train_balanced = train_balanced[FEATURE_COLS]
y_train_balanced = train_balanced[TARGET]

# 3. Scaling Fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test_raw)

print(f"Ukuran Training Set (Balanced): {X_train_scaled.shape}")
print(f"Ukuran Test Set (Steril): {X_test_scaled.shape}")

## 6. Fitur Unik 1: Custom Focal Loss Function

In [ ]:
def binary_focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        p_t = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        alpha_factor = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        focal_weight = alpha_factor * tf.pow((1.0 - p_t), gamma)

        loss = -focal_weight * tf.math.log(p_t)
        return tf.reduce_mean(loss)
    return focal_loss_fixed

print("Custom Focal Loss Function siap digunakan.")

## 7. Arsitektur Model Deep Neural Network (DNN)

In [ ]:
def build_dnn_model(input_dim):
    model = keras.Sequential([
        layers.Dense(128, input_dim=input_dim, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        layers.Dense(64, kernel_initializer='he_normal'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),

        layers.Dense(32, kernel_initializer='he_normal'),
        layers.Activation('relu'),

        layers.Dense(1, activation='sigmoid')
    ])
    return model

model_dnn = build_dnn_model(X_train_scaled.shape[1])
model_dnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=binary_focal_loss(gamma=2.0, alpha=0.25),
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Recall(name='recall')]
)
model_dnn.summary()

## 8. Pelatihan Model DNN dengan Callbacks

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    keras.callbacks.ModelCheckpoint('best_dnn_fraud_model.keras', monitor='val_loss', save_best_only=True)
]

history = model_dnn.fit(
    X_train_scaled, y_train_balanced,
    validation_split=0.2,
    epochs=25,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

## 9. Evaluasi & Fitur Unik 2 (Confidence Thresholding Adjustment)

In [ ]:
y_pred_proba = model_dnn.predict(X_test_scaled).ravel()

def evaluate_threshold(y_true, y_probs, threshold=0.70):
    preds = (y_probs >= threshold).astype(int)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    cm = confusion_matrix(y_true, preds)

    print(f"=== Evaluasi Confidence Threshold: {threshold:.2f} ===")
    print(f"Precision : {prec:.4f}")
    print(f"Recall    : {rec:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)

for th in [0.50, 0.70, 0.85]:
    evaluate_threshold(y_test, y_pred_proba, threshold=th)
    print("-"*40)

## 10. Ekspor Artefak Model untuk Deployment

In [ ]:
# Menyimpan model Keras & Scaler
model_dnn.save('talkingdata_dnn_model.keras')
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Artefak model 'talkingdata_dnn_model.keras' dan 'scaler.pkl' berhasil disimpan!")